# M5 Forecasting — Model Training (XGBoost Baseline)

**Project:** Retail-Demand-Forecasting
**Stage:** `06_model_training` — preprocessing, baseline model, evaluation, artifacts
**Status:** Experimental model-training notebook — not a production training pipeline
**Author:** Data Engineering / Analytics Team

---

## Where this notebook sits in the project

```
Raw data → ETL → PostgreSQL → analytical subset → feature engineering
    │
    ▼
data/processed/features/{train,validation,test}.parquet   (built by 05_feature_engineering.ipynb)
    │
    ▼
[ THIS NOTEBOOK ]
    │
    ├─ Load train/validation/test (already split chronologically — not re-split here)
    ├─ Separate X / y, identify numerical vs. categorical columns
    ├─ Build a preprocessing pipeline (OneHotEncoder for categoricals), fit on TRAIN ONLY
    ├─ Train a seasonal-naive baseline (sales_lag_7) for context
    ├─ Train an XGBoost baseline, evaluate on validation
    ├─ Try one small, sensible improvement, compare on validation
    ├─ Evaluate the selected model ONCE on test
    ├─ Save model + preprocessor + metadata artifacts
    └─ Reload the artifacts and run a small inference sanity check
    │
    ▼
artifacts/{model, preprocessing}/*.pkl, artifacts/model_metadata.json
    │
    ▼
Later: inference / API / deployment (not part of this notebook)
```

## Objective

Establish a practical, honestly-evaluated XGBoost baseline for `sales_quantity`, built on top
of the already-validated, leakage-safe feature set from `05_feature_engineering.ipynb` — and
demonstrate that the resulting model is a *usable artifact* (savable, reloadable, capable of
producing a prediction), not just a notebook variable that disappears when the kernel closes.

### Explicit scope boundaries

- **No re-splitting of the data.** The train/validation/test Parquet files were already split
  chronologically in the previous notebook; this notebook loads them as-is.
- **No large hyperparameter search.** One baseline, one small, deliberate improvement, compared
  honestly on validation.
- **No target encoding, no embeddings, no scaling** — none of these are needed for the planned
  model (XGBoost) or the current categorical cardinality (see Sections 7–8).
- **No `src/` refactor.** This notebook is the experimental stage; reusable modules come later,
  once this design is confirmed (matching the same notebook → validate → refactor pattern
  already used for ETL).
- **The test set is touched exactly once**, after the final model is already selected on
  validation performance.


## 2. Imports

Standard `pandas`/`numpy` for data handling, `scikit-learn` for the preprocessing pipeline and
evaluation metrics, `xgboost` for the model, `joblib` for artifact persistence, and `json`/
`pathlib`/`time` for configuration, paths, and timing.


In [1]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

print(f"pandas version   : {pd.__version__}")
print(f"scikit-learn ver.: {__import__('sklearn').__version__}")
print(f"xgboost version   : {xgb.__version__}")

RANDOM_STATE = 42


pandas version   : 3.0.5
scikit-learn ver.: 1.9.0
xgboost version   : 3.4.0


**What we're doing:** importing the libraries this notebook needs and fixing a single
`RANDOM_STATE` used everywhere a seed is needed (model training, any randomized search), so the
notebook's results are reproducible from one run to the next.


## 3. Configuration and Paths

**What we're doing:** resolving the project root with the same marker-based `pathlib` search
used in every previous notebook, then defining every input/output path this notebook needs —
the three feature Parquet files, their metadata, and the `artifacts/` output directories.

**Why relative paths:** this notebook is assumed to run from within the project (as every
notebook in this project does) — no absolute paths, Windows or otherwise, appear anywhere
below.


In [2]:
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "data" / "processed" / "features").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate 'data/processed/features' above {current}. "
        "Run this notebook from inside the Retail-Demand-Forecasting project, "
        "after 05_feature_engineering.ipynb has produced the feature Parquet files."
    )

FEATURES_DIR = project_root / "data" / "processed" / "features"
TRAIN_PATH = FEATURES_DIR / "train.parquet"
VALID_PATH = FEATURES_DIR / "validation.parquet"
TEST_PATH = FEATURES_DIR / "test.parquet"
METADATA_PATH = FEATURES_DIR / "metadata.json"

ARTIFACTS_DIR = project_root / "artifacts"
MODEL_DIR = ARTIFACTS_DIR / "model"
PREPROCESSING_DIR = ARTIFACTS_DIR / "preprocessing"
MODEL_METADATA_PATH = ARTIFACTS_DIR / "model_metadata.json"

for path in [TRAIN_PATH, VALID_PATH, TEST_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Expected input file not found: {path}")

print(f"Project root      : {project_root}")
print(f"Features directory : {FEATURES_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR} (will be created when artifacts are saved)")


Project root      : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting
Features directory : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\features
Artifacts directory: D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\artifacts (will be created when artifacts are saved)


**What the result tells us:** every input file this notebook depends on is confirmed
present *before* anything is loaded — a missing upstream file (e.g. `05_feature_engineering.ipynb`
not having been run yet) fails clearly here, rather than as a confusing error partway through
model training.


## 4. Load Train / Validation / Test Datasets

**What we're doing:** loading the three already-split Parquet files, plus their metadata JSON,
as three **separate** DataFrames. We do not concatenate them at any point in this notebook —
each one plays a distinct, non-interchangeable role (train fits the model and the
preprocessor; validation selects between candidate models; test is evaluated exactly once, at
the end).

**Why we don't re-split here:** the previous notebook already established the chronological
split, with a programmatically verified `max(train_date) < min(validation_date) <
max(validation_date) < min(test_date)` guarantee. Re-deriving that split here would duplicate
logic and risk drifting out of sync with the metadata this notebook also loads.


In [3]:
train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)
test_df = pd.read_parquet(TEST_PATH)

with open(METADATA_PATH) as f:
    feature_metadata = json.load(f)

print(f"train_df shape     : {train_df.shape}")
print(f"valid_df shape      : {valid_df.shape}")
print(f"test_df shape       : {test_df.shape}")

print(f"\nMetadata target column     : {feature_metadata['target_column']}")
print(f"Metadata identifier columns: {feature_metadata['identifier_columns']}")
print(f"Metadata feature columns ({len(feature_metadata['feature_columns'])}): {feature_metadata['feature_columns']}")
print(f"\nMetadata split dates: {json.dumps(feature_metadata['split_dates'], indent=2)}")


train_df shape     : (765000, 37)
valid_df shape      : (165000, 37)
test_df shape       : (165000, 37)

Metadata target column     : sales_quantity
Metadata identifier columns: ['item_id', 'store_id', 'd', 'date']
Metadata feature columns (26): ['day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year', 'is_event', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_lag_1', 'price_change', 'price_change_pct', 'relative_price', 'sales_lag_1', 'sales_lag_7', 'sales_lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'weekday', 'dept_id', 'cat_id', 'state_id']

Metadata split dates: {
  "train_start": "2013-01-01",
  "train_end": "2014-05-25",
  "validation_start": "2014-05-26",
  "validation_end": "2014-09-12",
  "test_start": "2014-09-13",
  "test_end": "2014-12-31"
}


**What the result tells us:** the three datasets loaded with the row counts and split
date boundaries already established by `05_feature_engineering.ipynb`. The feature column list
recorded in `metadata.json` is treated as informative context here — Section 7 still derives
the actual numerical/categorical column lists **from the loaded DataFrames themselves**, not by
trusting this list blindly, per the "use the actual columns found in the Parquet files" design
decision for this notebook.


## 5. Basic Validation

**What we're doing:** a focused set of checks before any modeling work: column compatibility
across the three splits, target presence, duplicate-key check, and the chronological ordering
guarantee — re-verified here independently, not assumed to still hold from the previous
notebook's own checks.


In [4]:
print("--- column compatibility across splits ---")
train_cols, valid_cols, test_cols = set(train_df.columns), set(valid_df.columns), set(test_df.columns)
print(f"train == validation columns: {train_cols == valid_cols}")
print(f"train == test columns      : {train_cols == test_cols}")
assert train_cols == valid_cols == test_cols, "Column mismatch between splits!"

TARGET_COL = feature_metadata["target_column"]
IDENTIFIER_COLS = feature_metadata["identifier_columns"]

print(f"\n--- target column presence ---")
for name, d in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    print(f"'{TARGET_COL}' in {name}: {TARGET_COL in d.columns}")
    assert TARGET_COL in d.columns

print("\n--- duplicate (item_id, store_id, date) rows ---")
for name, d in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    n_dupes = d.duplicated(subset=["item_id", "store_id", "date"]).sum()
    print(f"{name}: {n_dupes} duplicates")
    assert n_dupes == 0

print("\n--- chronological ordering across splits ---")
print(f"max(train date)      : {train_df['date'].max().date()}")
print(f"min(validation date) : {valid_df['date'].min().date()}")
print(f"max(validation date) : {valid_df['date'].max().date()}")
print(f"min(test date)       : {test_df['date'].min().date()}")
assert train_df["date"].max() < valid_df["date"].min(), "Train/validation date overlap!"
assert valid_df["date"].max() < test_df["date"].min(), "Validation/test date overlap!"
print("\nConfirmed: train dates < validation dates < test dates.")


--- column compatibility across splits ---
train == validation columns: True
train == test columns      : True

--- target column presence ---
'sales_quantity' in train: True
'sales_quantity' in validation: True
'sales_quantity' in test: True

--- duplicate (item_id, store_id, date) rows ---
train: 0 duplicates
validation: 0 duplicates
test: 0 duplicates

--- chronological ordering across splits ---
max(train date)      : 2014-05-25
min(validation date) : 2014-05-26
max(validation date) : 2014-09-12
min(test date)       : 2014-09-13

Confirmed: train dates < validation dates < test dates.


**What the result tells us:** all three splits share an identical column set, the
target is present everywhere it needs to be, no duplicate observation keys exist in any split,
and the chronological non-overlap guarantee established in the previous notebook still holds
for these specific files. This is the green light to proceed to modeling.


## 6. Separate X and y

**What we're doing:** building the feature matrix `X` and target vector `y` for each split,
excluding the target and the identifier columns from `X` — while keeping the identifiers
available separately (`*_ids`) for later per-series analysis and prediction interpretation, per
the project's stated preference not to discard them entirely.

**Why identifiers are excluded from `X` at this stage specifically:** `item_id` and `store_id`
are high-cardinality strings; one-hot encoding them would multiply the feature matrix's column
count dramatically for comparatively little expected benefit in an MVP baseline (Section 8
revisits this explicitly). `d` and `date` are metadata, not intended as direct model inputs (the
calendar-derived features from `05_feature_engineering.ipynb` already capture the relevant time
signal).


In [5]:
FEATURE_COLS = [c for c in train_df.columns if c not in IDENTIFIER_COLS + [TARGET_COL]]

X_train, y_train = train_df[FEATURE_COLS].copy(), train_df[TARGET_COL].copy()
X_valid, y_valid = valid_df[FEATURE_COLS].copy(), valid_df[TARGET_COL].copy()
X_test, y_test = test_df[FEATURE_COLS].copy(), test_df[TARGET_COL].copy()

train_ids = train_df[IDENTIFIER_COLS].copy()
valid_ids = valid_df[IDENTIFIER_COLS].copy()
test_ids = test_df[IDENTIFIER_COLS].copy()

assert TARGET_COL not in X_train.columns
assert TARGET_COL not in X_valid.columns
assert TARGET_COL not in X_test.columns

print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape}, y_valid: {y_valid.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")
print("\nConfirmed: target column is absent from every X.")


Feature columns (32): ['wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'dept_id', 'cat_id', 'state_id', 'day_of_week', 'day_of_month', 'week_of_year', 'quarter', 'is_event', 'price_lag_1', 'price_change', 'price_change_pct', 'relative_price', 'sales_lag_1', 'sales_lag_7', 'sales_lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28']

X_train: (765000, 32), y_train: (765000,)
X_valid: (165000, 32), y_valid: (165000,)
X_test : (165000, 32), y_test : (165000,)

Confirmed: target column is absent from every X.


**What the result tells us:** six model-ready objects now exist (`X_train`/`y_train`,
`X_valid`/`y_valid`, `X_test`/`y_test`), plus the identifier columns preserved separately per
split — with an explicit assertion (not just a printed list) confirming the target cannot
appear inside any `X`.


## 7. Identify Numerical and Categorical Feature Columns

**What we're doing:** classifying `FEATURE_COLS` into numerical vs. categorical based on the
**actual dtypes present in `X_train`** — not a hardcoded list — so this cell keeps working
correctly even if the upstream feature set changes slightly (e.g. a feature is added, renamed,
or removed in a future run of `05_feature_engineering.ipynb`).

**Why this matters here specifically:** hardcoding two fixed column lists would silently break
(or silently miscategorize a column) the moment the feature set drifts even slightly from what
was assumed while writing this notebook. Deriving the split from `X_train.dtypes` directly
keeps this cell correct by construction.


In [ ]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Include boolean columns as numerical features.
# SNAP indicators are True/False and can be passed directly to XGBoost.
numerical_cols = X_train.select_dtypes(
    include=[np.number, "bool"]
).columns.tolist()

unclassified = (
    set(FEATURE_COLS)
    - set(categorical_cols)
    - set(numerical_cols)
)

assert not unclassified, (
    f"Columns with an unexpected dtype, not classified as either: {unclassified}"
)

print(f"Categorical columns ({len(categorical_cols)}):")
print(categorical_cols)

print(f"\nNumerical / binary columns ({len(numerical_cols)}):")
print(numerical_cols)

AssertionError: Columns with an unexpected dtype, not classified as either: {'snap_CA', 'snap_TX', 'snap_WI'}

**What the result tells us:** the categorical columns detected this way should match
`dept_id`, `cat_id`, `state_id`, `weekday` — all low cardinality, confirmed directly above,
which is what makes one-hot encoding (Section 8) an appropriate, low-risk choice for them.
Every numerical feature is picked up automatically, whatever the exact current set is.


## 8. Build the Preprocessing Pipeline

**What we're doing:** building a `ColumnTransformer` with a `OneHotEncoder` branch for the
categorical columns and a pass-through branch for the numerical columns.

**Why `OneHotEncoder(handle_unknown="ignore")` specifically:** validation and test data can
contain categorical values that never appeared in training (e.g. a `dept_id` with no training
rows, in a smaller subset, or a truly novel category in a future data refresh). Without
`handle_unknown="ignore"`, the encoder would raise an error the first time this happens;
`"ignore"` instead encodes an unseen category as all-zeros for that column's one-hot block —
predictable, safe behavior rather than a pipeline crash.

**Why numerical features pass through unscaled:** as decided in the feature-engineering
report, XGBoost is a tree-based model — splits are threshold-based, not distance- or
gradient-magnitude-based, so `StandardScaler`/`MinMaxScaler` would add a step with no expected
benefit for this specific model. `"passthrough"` is used instead of an identity transformer to
avoid an unnecessary data copy for the numerical block.

**Why `item_id`/`store_id` are not part of this pipeline at all:** as decided in Section 6,
they're excluded from `X` entirely for this MVP baseline — not encoded, not passed through —
specifically because of their high cardinality relative to this dataset's size.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("numerical", "passthrough", numerical_cols),
    ],
    remainder="drop",
)

print("ColumnTransformer constructed:")
print(f"  categorical branch: OneHotEncoder(handle_unknown='ignore') on {len(categorical_cols)} columns")
print(f"  numerical branch  : passthrough on {len(numerical_cols)} columns")
print("\nNot yet fit -- fitting happens explicitly in the next section, on X_train only.")


**What the result tells us:** the pipeline object exists but has not touched any data
yet — `ColumnTransformer` construction is lazy, exactly like `sqlalchemy.create_engine()` was
lazy in the earlier Load-phase notebook. The next section is where fitting actually happens,
and it happens deliberately on training data only.


## 9. Fit Preprocessing — ON TRAINING DATA ONLY

**This is the most important leakage-prevention step in this notebook.**

**What we're doing:** calling `preprocessor.fit(X_train)` — and only `X_train`. The encoder's
learned category vocabulary comes exclusively from training data.

**Why this matters, concretely:** if the encoder were fit on `X_train` concatenated with
`X_valid` (or `X_test`), its internal state (which categories exist, in what order) would have
been informed by data the model is supposed to be evaluated *against* as if unseen. That's a
preprocessing-level leakage, distinct from — but just as serious as — the temporal leakage
already guarded against in feature engineering. The correct pattern, used here, is:

```
preprocessor.fit(X_train)                    # learn categories from training data ONLY
X_train_transformed = preprocessor.transform(X_train)
X_valid_transformed = preprocessor.transform(X_valid)   # transform only, never re-fit
X_test_transformed  = preprocessor.transform(X_test)    # transform only, never re-fit
```

**What we expect:** categories present only in validation/test (if any) will be encoded as
all-zero one-hot rows for that column's block, thanks to `handle_unknown="ignore"` — not an
error.


In [ ]:
fit_start = time.perf_counter()
preprocessor.fit(X_train)
fit_elapsed = time.perf_counter() - fit_start

print(f"preprocessor.fit(X_train) completed in {fit_elapsed:.3f}s")

ohe = preprocessor.named_transformers_["categorical"]
print(f"\nCategories learned per categorical column (from X_train only):")
for col, cats in zip(categorical_cols, ohe.categories_):
    print(f"  {col:<12}: {len(cats)} categories -> {list(cats)[:6]}{' ...' if len(cats) > 6 else ''}")


**What the result tells us:** the encoder's category lists were derived exclusively
from `X_train`. The next section transforms all three splits using this already-fit
preprocessor — `X_valid` and `X_test` are never involved in `.fit()`, only `.transform()`.


In [ ]:
print("--- sanity check: categories present in validation/test but NOT in training ---")
any_unseen = False
for col in categorical_cols:
    train_categories = set(X_train[col].unique())
    valid_unseen = set(X_valid[col].unique()) - train_categories
    test_unseen = set(X_test[col].unique()) - train_categories
    if valid_unseen or test_unseen:
        any_unseen = True
        print(f"{col}: unseen in validation = {valid_unseen or '{}'}, unseen in test = {test_unseen or '{}'}")

if not any_unseen:
    print("No unseen categories found in validation/test for this run -- "
          "handle_unknown='ignore' is a safeguard for future data, not something triggered here.")


**What the result tells us:** whether or not this particular run happens to contain a
truly unseen category, the pipeline is defensively correct either way — `handle_unknown="ignore"`
means this notebook doesn't depend on getting lucky with category overlap between splits.


## 10. Transform Train / Validation / Test

**What we're doing:** applying the already-fit preprocessor to all three splits via
`.transform()` — never `.fit_transform()` on anything but `X_train`.

**Memory note:** each transform call produces one new array per split — we do not additionally
retain multiple redundant transformed copies of the same split (e.g. no separate "scaled" and
"unscaled" versions, since scaling isn't used at all here).


In [ ]:
X_train_transformed = preprocessor.transform(X_train)
X_valid_transformed = preprocessor.transform(X_valid)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_valid_transformed shape: {X_valid_transformed.shape}")
print(f"X_test_transformed shape : {X_test_transformed.shape}")

# Get output feature names for later inspection / model feature importance.
transformed_feature_names = preprocessor.get_feature_names_out().tolist()
print(f"\nTotal transformed feature count: {len(transformed_feature_names)}")


**What the result tells us:** all three transformed matrices share the same number of
columns — confirmed explicitly in Section 11 — which is what makes it valid to train on one and
evaluate on the others.


## 11. Inspect Transformed Feature Dimensions

**What we're doing:** explicitly checking that one-hot encoding didn't produce a
surprisingly large feature matrix, and that all three transformed splits have identical column
counts (a hard requirement — a model trained on one column layout cannot predict on another).


In [ ]:
print(f"Original feature count (pre-encoding) : {len(FEATURE_COLS)} "
      f"({len(numerical_cols)} numerical + {len(categorical_cols)} categorical)")
print(f"Transformed feature count (post-encoding): {X_train_transformed.shape[1]}")

dims_match = (X_train_transformed.shape[1] == X_valid_transformed.shape[1] == X_test_transformed.shape[1])
print(f"\nTransformed dimensions match across train/validation/test: {dims_match}")
assert dims_match, "Transformed feature matrices have inconsistent dimensions across splits!"

expansion_factor = X_train_transformed.shape[1] / len(FEATURE_COLS)
print(f"Dimensionality expansion from one-hot encoding: {expansion_factor:.2f}x")
if expansion_factor > 3:
    print("NOTE: expansion factor is notably large -- worth revisiting categorical cardinality "
          "if this grows further as the project scales.")
else:
    print("Expansion factor is modest -- consistent with the low cardinality confirmed in Section 7.")


**What the result tells us:** the transformed matrices are dimensionally compatible
across all three splits, and the one-hot expansion is modest given the confirmed low
cardinality of `dept_id`/`cat_id`/`state_id`/`weekday` — exactly the outcome the "don't encode
`item_id`/`store_id`" decision in Section 6 was meant to protect against.


## 12. Seasonal-Naive Baseline (`sales_lag_7`)

**What we're doing:** using `sales_lag_7` directly as a prediction — "demand today equals
demand exactly one week ago." This requires no model fitting at all; it's a heuristic, included
specifically to answer *"does XGBoost actually improve over a simple forecasting rule?"* rather
than assuming it does.

**Handling missing `sales_lag_7` values explicitly:** the first 7 rows of every
`(item_id, store_id)` series have `sales_lag_7 = NaN` (documented in the feature-engineering
notebook). For this baseline specifically — not for the XGBoost model, which handles `NaN`
natively — we exclude rows where `sales_lag_7` is missing from the baseline's own evaluation,
since a naive "predict the missing value" rule has no sensible definition. This is a
deliberate, documented choice, not an oversight.


In [ ]:
def evaluate_naive_baseline(df_split, split_name):
    valid_mask = df_split["sales_lag_7"].notna()
    n_excluded = (~valid_mask).sum()
    y_true = df_split.loc[valid_mask, TARGET_COL]
    y_pred = df_split.loc[valid_mask, "sales_lag_7"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f"[{split_name}] naive baseline (predict = sales_lag_7): "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}  "
          f"({n_excluded} rows excluded for missing sales_lag_7, out of {len(df_split)})")
    return {"mae": mae, "rmse": rmse, "n_excluded": int(n_excluded), "n_evaluated": int(valid_mask.sum())}

naive_valid_metrics = evaluate_naive_baseline(
    valid_df,
    "validation"
)


**What the result tells us:** this is the bar the XGBoost model needs to clear to be
worth the additional complexity. If XGBoost's validation MAE/RMSE (Section 14) were *worse*
than this naive rule, that would be a strong signal something in the modeling setup is wrong —
not that XGBoost is fundamentally a bad choice.


## 13. Train the XGBoost Baseline

**What we're doing:** training a first `XGBRegressor` with a sensible, deliberately modest
configuration — not tuned, not exhaustive — appropriate for a ~26K-row development-scale
dataset (or the full ~1.1M-row subset in the real project run) on a normal laptop.

**Configuration reasoning:**
- `n_estimators=200`, `max_depth=6`, `learning_rate=0.1` — standard, moderate starting values;
  not chosen via search.
- `objective="reg:squarederror"` — the standard regression objective, appropriate for a
  continuous (if non-negative, integer-valued) demand target.
- `random_state=RANDOM_STATE` — reproducibility.
- `n_jobs=-1` — use available CPU cores; no distributed/GPU training introduced.


In [ ]:
baseline_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

train_start = time.perf_counter()
baseline_model.fit(X_train_transformed, y_train)
baseline_train_time = time.perf_counter() - train_start

print(f"Baseline XGBoost trained in {baseline_train_time:.2f}s "
      f"on {X_train_transformed.shape[0]:,} rows x {X_train_transformed.shape[1]} features.")


**What the result tells us:** the model trained successfully on the preprocessed
training matrix. Training time is recorded here specifically so it can be compared against the
"improved" configuration in Section 15 — not just for its own sake.


## 14. Validation Evaluation

**What we're doing:** evaluating the baseline model on the **validation** set only — the test
set remains untouched (Section 18 restates this explicitly). We compute MAE, RMSE, and WAPE.

**Why WAPE instead of MAPE:** MAPE (mean absolute percentage error) divides by the actual
value for every row — for a target with many legitimate zero-sales observations (documented
extensively in the feature-engineering notebook and report), that produces division-by-zero or
extreme percentage errors on exactly the rows where demand is zero, which are common and
meaningful in this dataset. **WAPE (weighted absolute percentage error)** instead divides the
*sum* of absolute errors by the *sum* of actual values across the whole evaluated set:

```
WAPE = sum(|y_true - y_pred|) / sum(y_true)
```

This avoids the zero-denominator problem entirely (as long as the *sum* of `y_true` over the
evaluation set is non-zero, which it is here) and gives one interpretable "percentage of total
volume missed" figure across the whole series.


In [ ]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom


def evaluate_model(model, X, y, split_name, model_name):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    wape_score = wape(y, y_pred)
    print(f"[{split_name}] {model_name}: MAE={mae:.4f}, RMSE={rmse:.4f}, WAPE={wape_score:.4f}")
    return {"mae": mae, "rmse": rmse, "wape": wape_score}


baseline_valid_metrics = evaluate_model(baseline_model, X_valid_transformed, y_valid, "validation", "XGBoost baseline")


**What the result tells us:** this is the baseline model's honest, first look at
unseen (validation) data — the reference point every later comparison in this notebook (naive
baseline, improved model) is measured against.


## 15. A Small, Deliberate Model Improvement

**What we're doing:** trying **one** alternative configuration — not a grid search — chosen
from the standard XGBoost regularization/robustness knobs (`subsample`, `colsample_bytree`,
`min_child_weight`), evaluated the same way as the baseline, on validation only.

**Reasoning for this specific change:** the baseline uses no row/column subsampling
(`subsample=1.0`, `colsample_bytree=1.0` implicitly) and no minimum child weight constraint —
settings that can make a tree-based model more prone to overfitting individual training rows.
Adding modest subsampling and a small `min_child_weight` is a standard, low-risk robustness
adjustment — not a claim that this specific configuration is optimal, just a reasonable single
next step for an MVP.


In [ ]:
improved_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

train_start = time.perf_counter()
improved_model.fit(X_train_transformed, y_train)
improved_train_time = time.perf_counter() - train_start

print(f"Improved XGBoost trained in {improved_train_time:.2f}s")

improved_valid_metrics = evaluate_model(improved_model, X_valid_transformed, y_valid, "validation", "XGBoost improved")


**What the result tells us:** a second, honestly-comparable validation score, computed
identically to the baseline's — the comparison table in the next section makes the actual
decision, rather than assuming the "improved" configuration is automatically better just
because it has more regularization.


## 16. Model Comparison and Selection

**What we're doing:** consolidating every candidate's validation performance — the naive
seasonal baseline, the XGBoost baseline, and the improved XGBoost configuration — into one
table, and selecting the final model **based on this validation comparison alone**. The test
set has not been touched by any model at this point.


In [ ]:
comparison_rows = [
    {"model": "Seasonal-naive (sales_lag_7)", "mae": naive_valid_metrics["mae"], "rmse": naive_valid_metrics["rmse"],
     "wape": np.nan, "train_time_s": 0.0},
    {"model": "XGBoost baseline", "mae": baseline_valid_metrics["mae"], "rmse": baseline_valid_metrics["rmse"],
     "wape": baseline_valid_metrics["wape"], "train_time_s": round(baseline_train_time, 2)},
    {"model": "XGBoost improved", "mae": improved_valid_metrics["mae"], "rmse": improved_valid_metrics["rmse"],
     "wape": improved_valid_metrics["wape"], "train_time_s": round(improved_train_time, 2)},
]
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

# Select the final model based on validation MAE -- the primary metric for this comparison.
xgb_candidates = {"XGBoost baseline": (baseline_model, baseline_valid_metrics),
                   "XGBoost improved": (improved_model, improved_valid_metrics)}
best_name = min(xgb_candidates, key=lambda name: xgb_candidates[name][1]["mae"])
final_model, final_valid_metrics = xgb_candidates[best_name]

print(f"\nSelected final model based on validation MAE: {best_name}")
print(f"Validation metrics of selected model: {final_valid_metrics}")


**What the result tells us:** the model selection decision made here is directly
readable from the printed comparison table, not asserted separately from it — whichever
XGBoost configuration has the lower validation MAE is selected as `final_model`, and that
selection is what proceeds to the one-time test evaluation next. (The naive baseline is
included in the table for context, not as a candidate for "final model" selection — it exists
to answer "is a model worth it at all," which the comparison table already shows plainly.)


## 17. Confirm the Final Model Selection

**What we're doing:** a short, explicit confirmation step — stating in code, not just in
narrative, which model and configuration is the one proceeding to test evaluation.


In [ ]:
print(f"Final model selected : {best_name}")
print(f"Final model params    : {final_model.get_params()}")
print(f"\nThis model was selected using VALIDATION performance only.")
print(f"The test set has not been evaluated by any model up to this point in the notebook.")


**What the result tells us:** this cell is a deliberate checkpoint — anyone reading
the notebook top-to-bottom can see, in one place, exactly which model and hyperparameters were
chosen and on what basis, immediately before the test set is touched for the first and only
time.


## 18. Final Test Evaluation (One-Time, Out-of-Sample)

**What we're doing:** evaluating `final_model` on the test set — the first and only time the
test set is used anywhere in this notebook. This is the honest, final estimate of how the
selected model performs on genuinely unseen, future-dated data.

**Why only once:** repeatedly evaluating the test set while iterating on modeling choices would
turn it into a second validation set by another name — the whole point of holding it out is
that no decision in this notebook was influenced by it.


In [ ]:
final_test_metrics = evaluate_model(
    final_model,
    X_test_transformed,
    y_test,
    "test",
    best_name
)

naive_test_metrics = evaluate_naive_baseline(
    test_df,
    "test"
)

print(
    f"\nTest date range: "
    f"{test_df['date'].min().date()} "
    f"to "
    f"{test_df['date'].max().date()}"
)

print(f"Test observations: {len(test_df):,}")

print("\nFinal test comparison:")
print(
    f"XGBoost  MAE={final_test_metrics['mae']:.4f}, "
    f"RMSE={final_test_metrics['rmse']:.4f}, "
    f"WAPE={final_test_metrics['wape']:.4f}"
)

print(
    f"Naive    MAE={naive_test_metrics['mae']:.4f}, "
    f"RMSE={naive_test_metrics['rmse']:.4f}"
)

print(
    "\nThis is the final out-of-sample evaluation. "
    "No further model changes should be made based on test results."
)

**What the result tells us:** the numbers reported here are this project's actual,
current out-of-sample performance for the selected model — not a number chosen after seeing
test performance and iterating further. Comparing directly against the naive baseline's test
performance (computed identically, on the same held-out period) answers the practical question
this notebook set out to answer: does the engineered feature set plus XGBoost meaningfully beat
a simple seasonal heuristic on genuinely unseen data.


## 19–21. Save Model, Preprocessor, and Metadata Artifacts

**What we're doing:** persisting three things under `artifacts/`: the fitted `final_model`
(`artifacts/model/xgb_model.pkl`), the fitted `preprocessor`
(`artifacts/preprocessing/preprocessor.pkl`), and a metadata JSON
(`artifacts/model_metadata.json`) recording enough information for a future notebook/service to
reload and use both correctly — without needing to re-run this entire notebook.

**Why `joblib` specifically:** it's the standard choice for persisting `scikit-learn`-compatible
Python objects (including the `ColumnTransformer`/`OneHotEncoder` pipeline and, via its
scikit-learn-compatible API, the `XGBRegressor`) — more efficient than plain `pickle` for
objects containing large NumPy arrays, which both of these do.

**Why the metadata JSON is kept small:** it records exactly what a consumer needs — model name,
feature/categorical/numerical column lists, validation and test metrics, and the split date
ranges used — not an elaborate experiment-tracking schema, consistent with this project's MVP
scope.


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSING_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "xgb_model.pkl"
preprocessor_path = PREPROCESSING_DIR / "preprocessor.pkl"

joblib.dump(final_model, model_path)
joblib.dump(preprocessor, preprocessor_path)

print(f"Saved model        -> {model_path}  ({model_path.stat().st_size / 1024:.1f} KB)")
print(f"Saved preprocessor -> {preprocessor_path}  ({preprocessor_path.stat().st_size / 1024:.1f} KB)")


In [ ]:
model_metadata = {
    "model_name": best_name,
    "model_type": "xgboost.XGBRegressor",
    "target_column": TARGET_COL,
    "feature_columns": FEATURE_COLS,
    "categorical_columns": categorical_cols,
    "numerical_columns": numerical_cols,
    "transformed_feature_count": int(X_train_transformed.shape[1]),
    "model_params": {k: v for k, v in final_model.get_params().items() if v is not None},
    "validation_metrics": final_valid_metrics,
    "test_metrics": final_test_metrics,
    "naive_baseline_validation_metrics": naive_valid_metrics,
    "naive_baseline_test_metrics": naive_test_metrics,
    "train_date_range": {"start": str(train_df["date"].min().date()), "end": str(train_df["date"].max().date())},
    "validation_date_range": {"start": str(valid_df["date"].min().date()), "end": str(valid_df["date"].max().date())},
    "test_date_range": {"start": str(test_df["date"].min().date()), "end": str(test_df["date"].max().date())},
    "dataset_sizes": {"train_rows": len(train_df), "validation_rows": len(valid_df), "test_rows": len(test_df)},
}

with open(MODEL_METADATA_PATH, "w") as f:
    json.dump(model_metadata, f, indent=2)

print(f"Saved metadata -> {MODEL_METADATA_PATH}")
print(json.dumps(model_metadata, indent=2, default=str))


**What the result tells us:** `artifacts/` now contains everything needed to reload
and use this model — confirmed directly in the next section, not just assumed from a
successful `joblib.dump()` call.


## 22–23. Reload Artifacts and Run an Inference Sanity Check

**What we're doing:** loading the just-saved model and preprocessor back from disk into fresh
variables (deliberately named differently from the originals, so there's no chance of
accidentally reusing the in-memory objects), then running them through a small number of real
rows from the test set — end to end, exactly as a future inference service would: raw feature
row → preprocessor → model → prediction.

**Why this matters:** saving a file successfully doesn't guarantee it's actually usable — this
step is direct proof the artifacts round-trip correctly and produce predictions consistent
with the in-memory model, not just that `joblib.dump()` didn't raise an error.


In [ ]:
reloaded_model = joblib.load(model_path)
reloaded_preprocessor = joblib.load(preprocessor_path)

print("Reloaded model and preprocessor from disk into fresh variables.")

# Take a small number of real rows the model has never been fit on differently than before --
# same X_test rows, but pushed through the RELOADED objects to prove they work standalone.
sample_raw = X_test.iloc[:5]
sample_true = y_test.iloc[:5].values

sample_transformed = reloaded_preprocessor.transform(sample_raw)
sample_predictions = reloaded_model.predict(sample_transformed)

sanity_check_df = pd.DataFrame({
    "item_id": test_ids.iloc[:5]["item_id"].values,
    "store_id": test_ids.iloc[:5]["store_id"].values,
    "date": test_ids.iloc[:5]["date"].values,
    "actual_sales_quantity": sample_true,
    "predicted_sales_quantity": sample_predictions,
})
display(sanity_check_df)

# Confirm the reloaded objects produce IDENTICAL predictions to the original in-memory objects.
original_predictions = final_model.predict(preprocessor.transform(sample_raw))
predictions_match = np.allclose(sample_predictions, original_predictions)
print(f"\nReloaded artifacts produce identical predictions to the original in-memory objects: {predictions_match}")
assert predictions_match, "Reloaded model/preprocessor predictions differ from the original -- investigate!"


**What the result tells us:** the reloaded preprocessor and model, used together on raw
feature rows they were never given in-memory, produce predictions numerically identical to the
original objects — direct, working proof this artifact pair is a genuinely usable unit, not
just two files that happen to exist. This is the concrete basis for the conceptual inference
flow this project is working toward:

```
Raw/new observation → feature engineering → preprocessor.pkl → xgb_model.pkl → prediction
```

— everything to the right of "feature engineering" in that flow is now demonstrated working
end-to-end from saved artifacts.


## 24. Final Summary

### What this notebook did

1. Loaded the already-split, already-validated train/validation/test feature sets — no
   re-splitting.
2. Derived numerical/categorical column lists from actual dtypes, not a hardcoded list.
3. Built a `ColumnTransformer` (`OneHotEncoder` for categoricals, passthrough for numerics),
   **fit on `X_train` only**, transform-only on validation/test — verified with an explicit
   category-overlap sanity check.
4. Trained a seasonal-naive baseline (`sales_lag_7`) for context.
5. Trained an XGBoost baseline, then one deliberately small, reasoned improvement — compared
   honestly on validation, with no test-set involvement.
6. Selected the final model from validation performance alone, evaluated it exactly once on
   test, and reported MAE/RMSE/WAPE.
7. Saved the model, preprocessor, and metadata as artifacts, then reloaded them and proved —
   with a numerical equality check, not just a successful load — that they reproduce identical
   predictions standalone.

### What remains explicitly out of scope, by design

- No large hyperparameter search (Section 15 was one deliberate comparison, not a sweep).
- No target/mean encoding or embeddings for categoricals.
- No `item_id`/`store_id` encoding in this MVP baseline.
- No scaling of numerical features (not needed for XGBoost).
- No `src/` refactor — this notebook is the experimental stage; reusable
  `src/features/`, `src/preprocessing/`, `src/models/`, `src/evaluation/` modules are planned
  for after this design is confirmed, following the same pattern already used for the ETL
  stage (`src/etl/`).
- No inference API/service — the sanity check in Section 22–23 demonstrates the artifacts
  *could* power one, not that one exists yet.

### Where the actual numbers live

This notebook's own output (Sections 12, 14, 15, 16, 18) and `artifacts/model_metadata.json`
are the source of truth for this run's actual MAE/RMSE/WAPE figures — intentionally not
restated as fixed values in this summary, so this markdown never drifts out of sync with a
re-run of the notebook against updated data.
